# Breakout 1B · Targeting: from signals to a score

Workshop **Data Driven Domestic Revenue Mobilization**, Bangkok, October 2026.

You look at three years of customs declarations the way a risk unit does: first the six fields
everything depends on, then the **signals** (one thing that looks unusual about a line, each read from
a precomputed table), then a **ladder** of four ways to rank lines (a rule, distance from normal, an
oddness score, a learning score) and what happens to a rule as it ages, two new ideas, and an exercise
where your group chooses five lines to inspect.

**How to use it.** Press ▶ on each cell from top to bottom, or *Runtime › Run all*. Change only the sliders and drop-down menus. The code is folded: double-click a title to see it. Every result ends with a **Finding** sentence.

Everything here is synthetic: every importer, broker and seller is generated. Nothing heavy runs here:
the signals and scores were computed once and the notebook reads them.

In [ ]:
#@title 0 · Where the data comes from
#@markdown Leave these as they are. Your analysts can point DATA_URL at their own extract (section 6).
DATA_URL = "https://raw.githubusercontent.com/FrancoisChastel/CustomsWorkshopNotebooks/data-v8/data"  #@param {type:"string"}
INSTALL = True  #@param {type:"boolean"}

In [ ]:
#@title 0 · Setup (run me first)
import sys, subprocess, warnings
warnings.filterwarnings("ignore")
if INSTALL and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/FrancoisChastel/CustomsWorkshopNotebooks@data-v8"], check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from workshop_methods import notebook as nb
from workshop_methods.data import load_json, load_tables
from workshop_methods import ladder, newideas, signals
nb.use_style()
pd.set_option("display.max_colwidth", 40)
CALIBRATION = "random draw finds something on 1 to 8 lines in a hundred; this set is enriched on the 20 exercise lines only."
def calibration_line():
    """Printed under every table that shows a rate."""
    display(Markdown(f"<small><i>Calibration: {CALIBRATION}</i></small>"))
HOME_USE = ("home_use", "warehouse_out")  # the regimes where duty is paid: the lines a price check ranks
T = load_tables(DATA_URL, ['features_lines', 'declarations_h1', 'declarations_h2', 'tariff', 'trader_features', 'importers', 'offices'])
print(f"Loaded {len(T)} tables from {DATA_URL}. Ready.")

In [ ]:
#@title 0 · Prepare the tables
f = T["features_lines"]                                   # one row per line: the signals and the scores
d = pd.concat([T["declarations_h1"], T["declarations_h2"]], ignore_index=True)
tariff, tf = T["tariff"].set_index("hs8"), T["trader_features"].set_index("importer_id")
home = f[f["regime"].isin(HOME_USE)]                     # the lines that pay duty: what a price check ranks
h1, h2 = f[f["period_month"] <= 18], f[f["period_month"] > 18]
draw = h2[h2["random_draw"] == 1]                        # months 19-36: controlled whatever they look like
def hit_rate(frame):
    """Share of findings among lines that carry an outcome (uncontrolled lines do not count)."""
    seen = frame["outcome"].dropna()
    return float((seen == "finding").mean()) if len(seen) else float("nan")
sheet = pd.DataFrame(load_json(DATA_URL, "worksheet_1b")["lines"]).set_index("line_ref")
print(f"{len(d):,} lines over {d['month'].nunique()} months ({d['month'].min()} to {d['month'].max()}), "
      f"{len(T['importers']):,} importers on the register, {d['broker_id'].nunique()} brokers, {len(T['offices'])} offices, "
      f"{len(tariff)} tariff lines. {len(home):,} lines pay duty (home use). The worksheet holds {len(sheet)} lines.")

## 1 · Look at the data

Six fields decide whether anything downstream can be computed: the **importer identifier**, the
**quantity and its unit**, the **description text**, the **value fields**, the **dates** and the
**outcomes**. One check each, printed on this set. Two of the six carry a trap on purpose.

In [ ]:
#@title 1 · Six fields, one check each
nb.steps("Can the six fields the rest depends on be trusted?",
         "Importer identifier: is the customs importer code there, and the tax ID behind it?",
         "Quantity and unit: is the unit the code's trade unit, so that the quantity means something?",
         "Description text: is there text a reader can place against a code?",
         "Value fields: do FOB, freight and insurance add up to the customs value, and is every value positive?",
         "Dates: do the lodgement dates sit inside the 36 months and agree with the month field?",
         "Outcomes: where a control took place, is an outcome recorded; where none did, is the field empty?")
unit_ok = d["unit"] == d["hs8"].map(tariff["unit"])
readable = f.set_index("line_id")["text_readable"].reindex(d["line_id"]).fillna(1).to_numpy()
adds_up = (d["fob_lcu"] + d["freight_lcu"] + d["insurance_lcu"] - d["customs_value_lcu"]).abs() <= 1
positive = (d["customs_value_lcu"] > 0) & (d["net_kg"] > 0)
in_window = d["decl_date"].between("2023-10-01", "2026-09-30") & (d["decl_date"].dt.strftime("%Y-%m") == d["month"])
controlled = d["control_type"].notna() & (d["control_type"] != "")
tin_blank = d["tin_customs"].fillna("").eq("").mean()
checks = pd.DataFrame([
    ["importer identifier", "customs code blank / tax ID blank", f"{(d['importer_id'] == '').mean():.1%} / {tin_blank:.1%}",
     "clean; the blank tax IDs are the informal importers (session 4B)"],
    ["quantity and unit", "unit is not the code's trade unit", f"{(~unit_ok).mean():.1%}", "a trap on purpose: value per unit is nonsense on these lines"],
    ["description text", "text a reader cannot place against any code", f"{(readable == 0).mean():.1%}", "a trap on purpose: abbreviations, brands, a second language"],
    ["value fields", "FOB + freight + insurance differs from the value, or a value is not positive", f"{(~adds_up | ~positive).mean():.1%}", "clean"],
    ["dates", "outside the 36 months, or month and date disagree", f"{(~in_window).mean():.1%}", "clean"],
    ["outcomes", "controlled lines without an outcome", f"{(controlled & d['outcome'].isna()).mean():.1%}", "clean; an empty outcome means no control took place"],
], columns=["field", "check", "lines failing", "verdict"]).set_index("field")
nb.show(checks)
nb.finding(f"Identifiers, values and dates are clean. {(~unit_ok).mean():.1%} of lines carry a unit that is not the code's trade "
           f"unit and {(readable == 0).mean():.1%} a description no reader can place: the two traps this set carries on purpose, "
           "and the two checks to run before any price rule. Read the six as a monthly quality number, not a one-off.")

In [ ]:
#@title 1 · Quantity against weight: the units trap
nb.steps("Do the quantity and the net weight tell the same story?",
         "Work out the weight each line should have from its quantity and unit (kg, litres, pieces, pairs, dozens x the code's usual kg per unit).",
         "Divide the declared net weight by that expected weight; a line far from 1 has its quantity in another unit.",
         "Work in value per kilo, never per unit: the kilo is the only price every line shares.")
ratio = home["net_kg"] / signals.expected_weight(home, T["tariff"])
off = ratio.notna() & ((ratio > 3) | (ratio < 1 / 3))
worst = home.assign(weight_ratio=ratio)[off].sort_values("weight_ratio", ascending=False).head(6)
nb.show(worst[["line_id", "importer_id", "hs8", "quantity", "unit", "net_kg", "weight_ratio"]],
        formats={"net_kg": "{:,.0f}", "quantity": "{:,.0f}", "weight_ratio": "{:,.1f}x"})
top = worst.iloc[0]
nb.finding(f"{off.mean():.1%} of home-use lines fail the check. Line {top['line_id']} declares {top['quantity']:,.0f} {top['unit']} but "
           f"{top['net_kg']:,.0f} kg, {top['weight_ratio']:,.0f} times what the quantity implies: its value per unit is nonsense while "
           "its value per kilo is normal. Drop these lines from every price check, or a price rule flags an honest trader.")

In [ ]:
#@title 1 · Two halves of time, and what an empty outcome means
nb.steps("What does each half of the 36 months carry?",
         "Months 1-18: the five legacy rules gave every line a lane (green, yellow, red); every yellow and red line and a 10% random draw were controlled.",
         "Months 19-36: no lanes; only the random draw, drawn before any rule and controlled in full.",
         "The outcome field is filled only where a control took place. Empty means 'not looked at', never 'no finding'.")
d1, d2 = d[d["period_month"] <= 18], d[d["period_month"] > 18]
lanes = d1["lane"].value_counts(normalize=True)
halves = pd.DataFrame({
    "lines": [len(d1), len(d2)],
    "lanes": [f"green {lanes.get('green', 0):.0%} · yellow {lanes.get('yellow', 0):.0%} · red {lanes.get('red', 0):.0%}", "none"],
    "random draw": [int((d1["random_draw"] == 1).sum()), int((d2["random_draw"] == 1).sum())],
    "lines with an outcome": [int(d1["outcome"].notna().sum()), int(d2["outcome"].notna().sum())],
    "outcome empty": [d1["outcome"].isna().mean(), d2["outcome"].isna().mean()],
    "findings where controlled": [hit_rate(d1), hit_rate(d2)],
}, index=["months 1-18", "months 19-36"])
nb.show(halves, formats={"lines": "{:,}", "random draw": "{:,}", "lines with an outcome": "{:,}", "outcome empty": "{:.0%}", "findings where controlled": "{:.1%}"})
calibration_line()
nb.finding(f"Months 1-18 carry the legacy lanes and {d1['outcome'].notna().sum():,} controlled lines with an outcome; months 19-36 carry "
           f"no lanes and only the {int((d2['random_draw'] == 1).sum()):,} lines of the random draw, which found something on "
           f"{hit_rate(d2):.1%} of them. The outcome is empty on {d['outcome'].isna().mean():.0%} of all lines: nothing was checked "
           "there, and a model must never read that as 'nothing wrong'.")

## 2 · Signals

A **signal** is one thing that looks unusual about a line. Every signal below was computed once for
every line (the `features_lines` table); each cell reads one, shows where it is strongest, and says
what it found. The peer group comes first because every price signal depends on it.

In [ ]:
#@title 2 · The peer group: who do we compare a line with?
CODE = "85171301"  #@param {type:"string"}
ORIGIN = "CHN"  #@param {type:"string"}
MIN_LINES = 30  #@param {type:"slider", min:10, max:60, step:5}
nb.steps(f"Which lines count as 'the same goods' for a line of code {CODE} from {ORIGIN}?",
         "Level 1: same 8-digit code and origin over the last 12 months (months 25 to 36). Level 2: same code, any origin. Level 3: same 6-digit heading, any origin.",
         f"Use the first level with at least {MIN_LINES} lines; take its median value per kilo and its spread (1.4826 x the median absolute deviation), in dollars at 25 LCU.",
         "A line's distance is (its value per kilo - the median) / the spread: how unusual, never what the goods are worth.")
window = home[home["period_month"] > 24]
usd = window["value_per_kg"] / 25
levels = {"1 · same code and origin": (window["hs8"] == CODE) & (window["origin"] == ORIGIN),
          "2 · same code, any origin": window["hs8"] == CODE,
          "3 · same heading, any origin": window["hs6"] == CODE[:6]}
rows, used = [], None
for name, mask in levels.items():
    v = usd[mask]
    med = v.median()
    rows.append({"peer group": name, "lines": int(len(v)), "median_usd_per_kg": med, "spread_usd_per_kg": 1.4826 * (v - med).abs().median()})
    used = used or (name if len(v) >= MIN_LINES else None)
table = pd.DataFrame(rows).set_index("peer group")
table["used"] = table.index == used
nb.show(table, formats={"median_usd_per_kg": "{:,.0f}", "spread_usd_per_kg": "{:,.0f}"}, highlight=table["used"])
cell = usd[levels[used]] if used else usd[levels["3 · same heading, any origin"]]
med, spread = cell.median(), 1.4826 * (cell - cell.median()).abs().median()
lowest = cell.idxmin()
z = (cell[lowest] - med) / spread
share = home["peer_level_used"].value_counts(normalize=True)
nb.finding(f"Code {CODE} from {ORIGIN}: {int(len(cell))} peers at level {used[0] if used else '3'} over the last 12 months, median "
           f"{med:,.0f} USD/kg, spread {spread:,.0f}. The lowest line, {window.at[lowest, 'line_id']} at {cell[lowest]:,.0f} USD/kg, "
           f"sits {z:.1f} spreads below the median: unusual, which says nothing yet about what the goods are worth. Across all "
           f"home-use lines {share.get(1, 0):.0%} are judged at level 1, {share.get(2, 0):.0%} at level 2, {share.get(3, 0):.0%} at "
           f"level 3 and {share.get(0, 0):.1%} have no peer group at any level: show the level beside every price flag.")

In [ ]:
#@title 2 · Price against peers: who is low again and again?
CUT = -3.0  #@param {type:"slider", min:-4, max:-2, step:0.5}
nb.steps("Which importers declare low values on many lines, not once?",
         "For every judged line, read its distance from the peer median in spreads (robust_z, median and spread, not the average).",
         f"Call a line low when it sits more than {abs(CUT):.0f} spreads below.",
         "For each importer with 15 lines or more, take the share of its lines that are low; read it beside its hit rate where it was controlled in months 1-18.")
judged = home[home["peer_level_used"] > 0]
low = (judged["robust_z"] < CUT).groupby(judged["importer_id"]).agg(["mean", "size"])
low = low[low["size"] >= 15].sort_values("mean", ascending=False)
table = low.head(8).rename(columns={"mean": "share_low", "size": "lines"})
table["median_share_of_peers"] = judged.groupby("importer_id")["share_of_median"].median().reindex(table.index)
table["hit_rate_h1"] = tf["hit_rate_h1"].reindex(table.index)
nb.show(table, formats={"share_low": "{:.0%}", "median_share_of_peers": "{:.2f}", "hit_rate_h1": "{:.0%}"})
calibration_line()
strong = table[table["share_low"] >= 0.25]
nb.finding(f"{(judged['robust_z'] < CUT).mean():.1%} of judged lines sit more than {abs(CUT):.0f} spreads below their peers; "
           f"{len(strong)} importers have a quarter or more of their lines there ({', '.join(strong.index[:4])}), against "
           f"{low['mean'].median():.1%} for the median importer. Repeated low prices point to a practice, not a one-off discount; "
           "the honest ones among them are contracted discounts and grade mixes, which the own-history check separates (section 3).")

In [ ]:
#@title 2 · Importer tenure: new names that grow fast
nb.steps("Which importers are both new and large?",
         "Read each importer's months since registration and its import value over the last 12 months (trader_features).",
         "Keep those registered less than 12 months ago; rank them by value and read their typical price against peers.")
new = tf[tf["tenure_months"] < 12].sort_values("value_12m_lcu", ascending=False)
view = new.head(6)[["tenure_months", "value_12m_lcu", "lines", "n_origins", "median_share_of_median"]]
nb.show(view, formats={"tenure_months": "{:.0f}", "value_12m_lcu": "{:,.0f}", "lines": "{:.0f}", "n_origins": "{:.0f}", "median_share_of_median": "{:.2f}"})
first = view.iloc[0]
rank = int((tf["value_12m_lcu"] > first["value_12m_lcu"]).sum()) + 1
nb.finding(f"{len(new)} importers ({len(new) / len(tf):.0%}) registered within the last 12 months. Importer {view.index[0]} registered "
           f"{first['tenure_months']:.0f} months ago and already ranks #{rank} of {len(tf):,} by import value, at {first['median_share_of_median']:.0%} "
           "of the typical price. Large volumes from a new name are the exception a rule needs: a first-year review, not a lane.")

In [ ]:
#@title 2 · Drift toward cheaper codes
nb.steps("Who is moving its goods toward the cheapest code of a heading, month after month?",
         "For each importer and broker, the precomputed drift: the rise over the trailing six months of the share of the pair's lines under the heading's cheapest code (0 when no rise).",
         "Keep pairs with 12 lines or more and read the largest rises; the broker is often the hand that moves them.")
pairs = home.groupby(["importer_id", "broker_id"]).agg(lines=("line_id", "size"), drift=("code_drift_score", "max"))
pairs = pairs[pairs["lines"] >= 12].sort_values("drift", ascending=False)
view = pairs.head(8).reset_index()
nb.show(view, formats={"drift": "{:+.0%}"})
movers = pairs[pairs["drift"] >= 0.30]
first = view.iloc[0]
nb.finding(f"Importer {first['importer_id']} with broker {first['broker_id']} moved its lines toward the cheapest code of the heading by "
           f"{first['drift']:.0%} over six months; {len(movers)} importer-broker pairs move by 30 points or more. Line by line nothing "
           "looks wrong; over a year the shift is plain. Ask the top pairs for product literature.")

In [ ]:
#@title 2 · Description against code
nb.steps("Does the description read like the code declared, or like another one?",
         "For each line, the precomputed disagreement: how much better another code's wording fits the description than the declared code's (character n-grams).",
         "Read the 20 lines where the disagreement is largest; keep those where the better-fitting code carries more duty.",
         "Group the rest by office: a vague description is a data-quality problem before it is a compliance problem.")
readable = home[home["text_readable"] == 1]
top = readable.sort_values("text_code_disagreement", ascending=False).head(20)
nb.show(top.head(8)[["line_id", "office", "hs8", "text_predicted_hs8", "text_higher_duty", "description"]],
        highlight=top.head(8)["text_higher_duty"] == 1)
dearer = int(top["text_higher_duty"].sum())
by_office = home.groupby("office")["text_code_disagreement"].apply(lambda s: (s > 0).mean()).sort_values(ascending=False)
nb.finding(f"{dearer} of the 20 lines whose description disagrees most with the code read like goods of a dearer code: the value "
           f"and the weight look normal, only the words disagree. Office {by_office.index[0]} writes a description that disagrees "
           f"with the code on {by_office.iloc[0]:.0%} of its lines, against {by_office.median():.0%} for the median office: fix the keying first.")

In [ ]:
#@title 2 · An origin that makes sense
nb.steps("Is a preference claimed from a country that actually makes the goods?",
         "Flag preference claims whose origin is not among the code's producers (tariff.origin_producers).",
         "For each foreign seller behind those claims, count the unrelated importers it serves, and its reach over 12 months (exporter_importers_12m).")
producers = tariff["origin_producers"].str.split("|")
claims = home[(home["preference_claim"] == "FTA") & (home["exporter_id"] != "")]
makes = np.array([o in producers.get(c, []) for c, o in zip(claims["hs8"], claims["origin"])])
doubtful = claims[~makes]
sellers = doubtful.groupby("exporter_id").agg(importers=("importer_id", "nunique"), lines=("line_id", "size"),
                                              origin=("origin", lambda s: s.mode().iloc[0]), reach_12m=("exporter_importers_12m", "max"))
sellers = sellers.sort_values("importers", ascending=False)
nb.show(sellers.head(6))
two = sellers.head(2)
nb.finding(f"{len(doubtful):,} lines ({len(doubtful) / len(claims):.0%} of preference claims) name an origin that does not make the goods. "
           f"Sellers {two.index[0]} ({two['origin'].iloc[0]}) and {two.index[1]} ({two['origin'].iloc[1]}) sit behind {two['importers'].iloc[0]} "
           f"and {two['importers'].iloc[1]} unrelated importers each: verify their certificates with the issuing authority first. "
           "A hub that relabels origin removes the full duty on every line it touches.")

In [ ]:
#@title 2 · Procedure and status history
nb.steps("Does an importer often change its declarations on its own?",
         "Leave out lines amended after a finding.",
         "For each importer with 20 lines or more, take the share of lines amended or withdrawn on its own initiative.")
hist = signals.history(d)
hist = hist[hist["lines"] >= 20]
view = hist.head(5)[["lines", "amended_share", "withdrawn_share", "changed_share"]]
nb.show(view, formats={"amended_share": "{:.0%}", "withdrawn_share": "{:.0%}", "changed_share": "{:.0%}"})
nb.finding(f"Importer {view.index[0]} amends or withdraws {view['changed_share'].iloc[0]:.0%} of its lines on its own, against "
           f"{hist['changed_share'].median():.0%} for the median importer: declarations tested against the controls. "
           "Add the amendment rate to the trader profile the risk committee reads.")

In [ ]:
#@title 2 · Twenty lines, four signals
nb.steps("How do the signals look on the worksheet's twenty lines?",
         "Price (how far below the peer median), description (how much better another code fits), origin (a preference claimed) and weight (how far the weight per unit sits from the code's usual) per line.")
four = sheet[["importer_id", "hs8", "signal_price", "signal_description", "signal_origin", "signal_weight"]]
strong = (four["signal_price"] >= 0.3) | (four["signal_description"] >= 0.3) | (four["signal_origin"] >= 1) | (four["signal_weight"].abs() >= 0.25)
nb.show(four, formats={c: "{:.2f}" for c in ["signal_price", "signal_description", "signal_weight"]}, highlight=strong)
nb.finding(f"{int(strong.sum())} of the 20 lines show at least one strong signal, and no single signal marks them all: "
           "which is why the ladder combines them, and why a units or own-history explanation must be checked before a pick.")

## 3 · The ladder

Four ways to rank lines, each a step up from the one before: a **rule**, **distance from normal**,
an **oddness score**, and a **learning score**. Each step keeps what the previous one does well and
fixes one of its mistakes. Every hit rate below is read on lines that were controlled whatever they
looked like: the random draw of months 19-36.

In [ ]:
#@title 3 · Step 1, a rule: the legacy price rule R2 and its decay
nb.steps("What does the legacy price rule find, and does it still work?",
         "Rule R2 (legacy_rules.md): red lane when value per kilo, in dollars, is below 70% of the reference price set in 2022 for the code (tariff.static_reference_2022; origin-blind, never refreshed).",
         "Months 1-18: the rule ran and every red line was controlled; read the red lane's hit rate.",
         "Months 19-36: the rule did not run; apply it to the random draw, where every line was controlled whatever it looked like, and read its hit rate again.")
reference = f["hs8"].map(tariff["static_reference_2022"])
R2 = (f["value_per_kg"] / 25) < 0.7 * reference
red = h1[h1["lane"] == "red"]
table = pd.DataFrame({
    "lines flagged by R2": [R2[h1.index].mean(), R2[draw.index].mean()],
    "hit rate on flagged lines (where controlled)": [hit_rate(h1[R2[h1.index]]), hit_rate(draw[R2[draw.index]])],
    "hit rate, red lane / random draw": [hit_rate(red), hit_rate(draw)],
}, index=["months 1-18 (legacy lanes)", "months 19-36 (random draw)"])
nb.show(table, formats={c: "{:.1%}" for c in table.columns})
calibration_line()
nb.finding(f"In months 1-18 the red lane found something on {hit_rate(red):.0%} of its lines, against {hit_rate(h1[h1['random_draw'] == 1]):.0%} "
           f"in the random draw of the same months. Applied to the months 19-36 random draw, R2 flags {R2[draw.index].mean():.0%} of lines and "
           f"finds something on {hit_rate(draw[R2[draw.index]]):.0%} of them: a reference that no longer moves with the market decays within "
           "a year (the expiry cell below shows the month). A rule is easy to defend and easy to fool; it needs a date and a control group.")

In [ ]:
#@title 3 · Step 2, distance from normal: against peers and against itself
Z = -3.0  #@param {type:"slider", min:-5, max:-1.5, step:0.5}
nb.steps("Is a low line far from normal, for its peers and for the importer itself?",
         "Distance from the peers: robust_z, how many spreads below the peer median (precomputed).",
         "Distance from itself: the same, against the importer's own earlier lines in the same code (months 1-28), for lines of months 29-36.",
         f"Keep importer-code pairs that look normal to their peers (80% to 125% of the median) and read the lowest own-history distance; flag below {Z}.")
recent, earlier = home[home["period_month"] >= 29], home[home["period_month"] <= 28]
own_z = signals.own_history_z(recent, earlier)
normal = recent["share_of_median"].between(0.8, 1.25) & own_z.notna()
pairs = pd.DataFrame({"own_z": own_z[normal], "share_of_peers": recent.loc[normal, "share_of_median"]}).groupby([recent.loc[normal, "importer_id"], recent.loc[normal, "hs8"]])
table = pairs.agg(recent_lines=("own_z", "size"), own_history_z=("own_z", "median"), share_of_peer_median=("share_of_peers", "median"))
table = table[table["recent_lines"] >= 10].sort_values("own_history_z")
nb.show(table.head(6).reset_index(), formats={"own_history_z": "{:.1f}", "share_of_peer_median": "{:.2f}"})
who, code_ = table.index[0]
first = table.iloc[0]
nb.finding(f"Importer {who} on code {code_}: its recent lines sit at {first['share_of_peer_median']:.0%} of the peer median, normal for everyone "
           f"else, but {abs(first['own_history_z']):.1f} spreads below its own earlier lines in the same code: a premium buyer now declaring at the "
           f"average. {int((table['own_history_z'] < Z).sum())} importer-code pairs sit below {Z} on their own history while looking normal to "
           "their peers: keep each importer's own band next to the peer band on the officer's screen.")

In [ ]:
#@title 3 · Step 3, an oddness score with its reasons
nb.steps("Which lines are odd on several counts at once?",
         "The precomputed oddness score (an isolation forest on the signals, no outcomes used): a line is odd when few lines look like it.",
         "Read the 20 oddest lines beside their single signals; count how many a single signal's top 20 would have missed.",
         "Read its hit rate at a 5% budget on the random draw, to compare with the next step.")
top = home.nlargest(20, "oddness_score")
view = top.head(8)[["line_id", "importer_id", "hs8", "robust_z", "text_code_disagreement", "weight_per_unit_dev", "oddness_score"]]
nb.show(view, formats={"robust_z": "{:.1f}", "text_code_disagreement": "{:.2f}", "weight_per_unit_dev": "{:+.2f}", "oddness_score": "{:.2f}"})
single = (set(home.nsmallest(20, "robust_z")["line_id"]) | set(home.nlargest(20, "text_code_disagreement")["line_id"])
          | set(home["weight_per_unit_dev"].abs().nlargest(20).index.map(home["line_id"])))
missed = len(set(top["line_id"]) - single)
odd_hit = ladder.hit_rate_at_budget(draw["oddness_score"], draw["outcome"] == "finding", 0.05)
nb.finding(f"{missed} of the 20 oddest lines are in no single signal's top 20: the score sees combinations, here an extreme price together with a "
           f"weight that disagrees with the quantity (the units trap again). At a 5% budget on the months 19-36 random draw its top lines carry a "
           f"finding {odd_hit:.0%} of the time: better than chance, far below a score that learned from outcomes. Show the reasons on every "
           "scored line so the officer can challenge the machine.")

In [ ]:
#@title 3 · Step 4, a learning score: hit rate at a 5% budget
BUDGET = 5  #@param {type:"slider", min:1, max:10, step:1}
nb.steps("Does a score trained on past controls find more than the rules, at the same budget?",
         "The learned score was trained once on the months 1-18 controlled lines and their outcomes (gradient boosting), then written beside every line.",
         f"On the months 19-36 random draw, inspect the top {BUDGET}% by each method and read the share with a finding.",
         "Compare with the legacy rule R2 on the same draw, the red lane of months 1-18, and the draw itself.")
found = draw["outcome"] == "finding"
rates = {"random draw": found.mean(), "legacy rule R2 (months 19-36)": hit_rate(draw[R2[draw.index]]), "legacy red lane (months 1-18)": hit_rate(red),
         "oddness score": ladder.hit_rate_at_budget(draw["oddness_score"], found, BUDGET / 100),
         "learned score": ladder.hit_rate_at_budget(draw["learned_score"], found, BUDGET / 100)}
fig, ax = nb.chart(f"At a {BUDGET}% budget the learned score finds something on {rates['learned score']:.0%} of the lines it picks")
ax.barh(list(rates)[::-1], [v * 100 for v in list(rates.values())[::-1]],
        color=[nb.COLORS["accent"], nb.COLORS["pale"], nb.COLORS["muted"], nb.COLORS["muted"], nb.COLORS["muted"]])
ax.set_xlabel("lines with a finding, %"); plt.show()
calibration_line()
nb.finding(f"On {len(draw):,} random-draw lines of months 19-36, the learned score finds something on {rates['learned score']:.0%} of its top "
           f"{BUDGET}%, the legacy rule R2 on {rates['legacy rule R2 (months 19-36)']:.0%} of the lines it flags, the random draw on "
           f"{rates['random draw']:.0%} (the red lane of months 1-18: {rates['legacy red lane (months 1-18)']:.0%}). "
           "In real administrations a random draw finds something on 1 to 8 lines in a hundred.")

In [ ]:
#@title 3 · Why the random draw matters: a score trained on red-lane lines only
nb.steps("What does a score learn when it only sees the lines the rules chose to open?",
         "The same model trained on red-lane lines only (learned_score_red_only) beside the one trained on every controlled line.",
         f"At the {BUDGET}% budget on the random draw: the hit rate of each, and what kind of finding each one catches.")
k = max(1, int(round(len(draw) * BUDGET / 100)))
picks = {"trained on all controlled lines": draw.nlargest(k, "learned_score"), "trained on red-lane lines only": draw.nlargest(k, "learned_score_red_only")}
kinds = ["undervaluation", "misclassification", "origin", "quantity"]
table = pd.DataFrame({name: {"hit rate": hit_rate(p), **{kind: int((p["finding_type"] == kind).sum()) for kind in kinds}} for name, p in picks.items()}).T
nb.show(table, formats={"hit rate": "{:.0%}", **{kind: "{:.0f}" for kind in kinds}})
calibration_line()
full, red_only = table["hit rate"].iloc[0], table["hit rate"].iloc[1]
lost = table.loc[table.index[0], "misclassification"] - table.loc[table.index[1], "misclassification"]
nb.finding(f"Trained on red-lane lines only, the score finds something on {red_only:.0%} of its picks against {full:.0%}, and what it finds is "
           f"mostly price: it catches {lost:.0f} fewer misclassified lines because the price rule never opened one. A model learns only from "
           "the lines someone chose to open; the random share, drawn before the rules run, is what lets it learn anything new. Its size "
           "is a committee decision, not a technical one.")

In [ ]:
#@title 3 · Expiry: a rule's hit rate month by month
nb.steps("How does a rule age once it is written?",
         "On the random draw of months 19-36, month by month: the hit rate of the learned score's top 5% (one cut for the whole half), of the legacy rule R2, and of the draw itself.",
         "Read the shift around month 30: the world moved, the rules did not.")
cut = draw["learned_score"].quantile(0.95)
monthly = draw.groupby("period_month").apply(lambda g: pd.Series({
    "random draw": hit_rate(g), "learned score, top 5%": hit_rate(g[g["learned_score"] >= cut]), "legacy rule R2": hit_rate(g[R2[g.index]])}))
fig, ax = nb.chart("The scores written on months 1-18 lose part of their hit rate after month 30; the random draw does not")
for col, color in [("learned score, top 5%", nb.COLORS["accent"]), ("legacy rule R2", nb.COLORS["amber"]), ("random draw", nb.COLORS["muted"])]:
    ax.plot(monthly.index, monthly[col] * 100, marker="o", color=color, label=col)
ax.axvline(30.5, color=nb.COLORS["ink"], linestyle=":"); ax.set_xlabel("month of the window"); ax.set_ylabel("lines with a finding, %"); ax.legend(); plt.show()
calibration_line()
early, late = monthly.loc[19:24].mean(), monthly.loc[31:36].mean()
nb.finding(f"Months 19-24: the learned score's top 5% finds something on {early['learned score, top 5%']:.0%} of lines, the legacy rule on "
           f"{early['legacy rule R2']:.0%}; months 31-36: {late['learned score, top 5%']:.0%} and {late['legacy rule R2']:.0%}, while the random "
           f"draw stays near {late['random draw']:.0%}. Every rule and every score needs an expiry date and a retraining plan, and the random "
           "draw is what tells you when.")

## 3b · Two new ideas

In [ ]:
#@title 3b · Groups inside one tariff line: 200 smartphone lines
K = 2  #@param {type:"slider", min:2, max:4, step:1}
nb.steps("Does one tariff line hide different products?",
         "Take the 200 smartphone lines of one code (cluster_1b).", "Plot value per kilo against weight per unit.",
         f"Let the machine find {K} groups (k-means, recomputed here in a second), then find the line farthest from every group.")
phones = pd.DataFrame(load_json(DATA_URL, "cluster_1b")["lines"])
groups = newideas.tariff_line_groups(phones, k=K)
far = groups["distance_to_nearest_group"].idxmax()
fig, ax = nb.chart("One line is priced like an entry model but packed like a premium one")
palette = [nb.COLORS["accent"], nb.COLORS["muted"], nb.COLORS["pale"], nb.COLORS["navy2"]]
for g, part in groups.groupby("group"):
    ax.scatter(part["kg_per_unit"], part["usd_per_kg"], s=14, color=palette[g - 1], label=f"group {g}")
ax.scatter(groups.at[far, "kg_per_unit"], groups.at[far, "usd_per_kg"], s=220, facecolors="none",
           edgecolors=nb.COLORS["amber"], linewidths=2.5, label="fits no group")
ax.set_xlabel("weight per unit, kg"); ax.set_ylabel("value per kilo, USD"); ax.legend(); plt.show()
row = groups.loc[far]
nb.finding(f"Line {phones.at[far, 'line_id']} is declared at {row['usd_per_kg']:.0f} USD/kg, an entry-level price, with {row['kg_per_unit']:.3f} kg "
           f"per unit, a premium model's weight: it fits no group. With k={K} the groups are a choice, not a truth; the odd line stays odd. "
           "Split broad codes into product groups before setting reference prices.")

In [ ]:
#@title 3b · A trader against its own past
IMPORTER = "IMP00684"  #@param ["IMP00684", "IMP00591", "IMP00926", "IMP00448", "IMP01142"]

SHOW_REFERENCE = False  #@param {type:"boolean"}
nb.steps("Did an importer's price fall away from its own history?",
         "Take 36 months of the importer's value per kilo on its main code (drift_1b).",
         "Build its band from earlier months only (expanding median and spread, shifted one month, a floor on the spread so the first months stay quiet).",
         "Flag the months below the band; tick 'show reference' to see whether everyone's price fell too.")
series = next(s for s in load_json(DATA_URL, "drift_1b")["series"] if s["importer_id"] == IMPORTER)
months = newideas.own_band(pd.DataFrame(series["months"]).assign(importer_id=series["importer_id"], hs8=series["hs8"]))
fig, ax = nb.chart(f"Importer {IMPORTER}, code {series['hs8']}: value per kilo against its own band")
ax.plot(months["month"], months["uv_lcu_per_kg"], color=nb.COLORS["accent"], label="value per kilo")
ax.plot(months["month"], months["band_lower"], color=nb.COLORS["muted"], linestyle=":", label="lower edge of its own band")
below = months[months["below_band"]]
ax.scatter(below["month"], below["uv_lcu_per_kg"], color=nb.COLORS["amber"], zorder=3, label="below the band")
if SHOW_REFERENCE:
    ax.plot(months["month"], months["ref_median"], color=nb.COLORS["navy2"], linestyle="--", label="typical value, all importers")
ax.set_xticks(months["month"][::6]); ax.legend(); plt.show()
if len(below):
    first = below.iloc[0]
    depth = 1 - first["uv_lcu_per_kg"] / first["band_median"]
    ref_fall = 1 - first["ref_median"] / months["ref_median"].iloc[:6].mean()
    nb.finding(f"Importer {IMPORTER} falls below its own band in {first['month']}, {depth:.0%} under its usual level, and stays there for "
               f"{len(below)} months. Over the same months the typical value for everyone moved by {-ref_fall:+.0%}: "
               + ("the market fell too, so this is a world price, not an importer." if ref_fall > 0.1 else "the market did not move; the drop is this importer's."))
else:
    nb.finding(f"Importer {IMPORTER} stays inside its own band for all {len(months)} months: nothing to ask.")

## 4 · Exercise: choose five lines to inspect

Roles: the **analyst** works the sliders, the **officer** decides, the **committee** writes what it
would sign. Weigh the four signals, read the ranked table, check each pick for a units or own-history
explanation, then pick five lines and lock them. Nothing about outcomes is visible until the
facilitator's reveal. Remember the calibration line: this set is enriched on these 20 lines only.

In [ ]:
#@title 4 · Weigh the signals and rank the twenty lines
HINTS = ["Not every low price is a finding.", "Weigh price, description, weight and origin, then check each pick for a units or own-history explanation.", "Price alone catches the two decoys; add description and weight, then drop the wrong unit and the discount."]

PRICE = 1.0  #@param {type:"slider", min:0, max:2, step:0.25}
DESCRIPTION = 1.0  #@param {type:"slider", min:0, max:2, step:0.25}
ORIGIN = 0.5  #@param {type:"slider", min:0, max:2, step:0.25}
WEIGHT = 1.0  #@param {type:"slider", min:0, max:2, step:0.25}
HINT_LEVEL = 0  #@param {type:"slider", min:0, max:3, step:1}
score = (PRICE * sheet["signal_price"] + DESCRIPTION * sheet["signal_description"] + ORIGIN * sheet["signal_origin"]
         + WEIGHT * sheet["signal_weight"].abs())
ranked = sheet.assign(score=score).sort_values("score", ascending=False)
nb.show(ranked[["importer_id", "hs8", "description", "origin", "unit", "value_per_kg", "peer_median", "score"]],
        formats={"value_per_kg": "{:,.0f}", "peer_median": "{:,.0f}", "score": "{:.2f}"})
nb.hint(HINT_LEVEL, HINTS, "1b.exercise")
nb.finding(f"With these weights the top five are {', '.join(ranked.index[:5])}. Before locking, check each pick for a units problem "
           "(quantity against weight) or a price that is normal for that importer's own history.")

In [ ]:
#@title 4 · Pick five lines and lock them
#@markdown If the selection list does not show, type your five line references here instead.
MY_PICKS = ""  #@param {type:"string"}
import ipywidgets as widgets
LOCKED_PICKS = [p.strip().upper() for p in MY_PICKS.split(",") if p.strip()]
choose = widgets.SelectMultiple(options=list(ranked.index), rows=10, description="Your five")
lock = widgets.Button(description="Lock my picks", button_style="primary")
note = widgets.Output()
def on_lock(_):
    global LOCKED_PICKS
    LOCKED_PICKS = list(choose.value)
    with note:
        note.clear_output()
        print(f"Locked {len(LOCKED_PICKS)} lines: {', '.join(LOCKED_PICKS)}" + ("" if len(LOCKED_PICKS) == 5 else "  (choose exactly five)"))
lock.on_click(on_lock)
display(widgets.VBox([choose, lock, note]))
if LOCKED_PICKS:
    print(f"Picks typed in: {', '.join(LOCKED_PICKS)}")

## 5 · Reveal (facilitator)

The facilitator types the code. Without it, this cell prints nothing.

In [ ]:
#@title 5 · Reveal
REVEAL_CODE = ""  #@param {type:"string"}
SEALED = {"salt": "TNbxd3sVarS+UJfUbZWc7Q==", "data": "ZTG976EkhU0QLBhILcJz5pLoCWVTw7NBwxPoKz9/JcHPrTj6kOjlWpWgb/cJR4MdIBl6dE7kbNg0VHrQQGjA93ufH2GJMZOiETPbSORNYqivKluwi29g5OKd2QexuPEZfAP7ueaL6lVbkXJMhrmJ5u3VUyiCMsjszfDWReu0rojRn1Tdf1sf4NX235QdpduAqeDNQXfLh4los4oB04GYOBFg2SaOdO7YPyFLYNgB/PTHodq2wQV+AAR6JYV6KPQYc9Fv+SZgB1Ie4EqQ9lyj/LYL5EbFB+gXsydjzTBY3jUPfIyX0g3wrrSqWpWUxb4SSkh9tCIFYk2vfZbZoR1QyJfXUrb3AGNFhxi+I0Vut88Js0CAWdlo64Qmzf9I3sNfMkfgFgGjb+3/kPBNA3LTpNak3GJsX7QsAWdqZqozv80LHm3luuKjAQE6++a7Ew3M/YjE9wKLVA4FygOA4yNXUtWr91mYvytmqsNVl0zHzQfpAIwNm7aKgtOciQS43ijKVQZ+PgeEA6Q3bV4N5At9agkujUPdZ+2x7DRFQ416WeX6iCnXK0w4Oz7bgit5+JQdJl4F6DwDe0hCDuAfuwye4ExwJw1iZJ+fRo3UejJZZGsiPr5WCuCa5GOiUS2SWSCCI0C68//ZT8BEocTD5uY+OHMjitkU6KidoXEBHDTBVAt8QPPqV5NKyT96LKhzIQNkDvC4aEUyBJqV3YmLGluUO1t6YUFcFhvxGWqVLgW/gAzGO4AQI9z9GmYJKe8q1qGHI7AQdLKfLtGNuc4H+7K1mnry2Ve/3Im1a3J/UD2Hw1UIef3W9gprT1AqLcI6BGav2ilGBMwm8cVorPQksdgZ9CT5j7eeprHeIUIKBgrCzoHJMA==", "mac": "890ab29ae9020111c08fa90932be42809f0973de51af5b259fac1e1d8a7c2862"}
answer = nb.unseal(REVEAL_CODE, SEALED)
if answer:
    key = pd.DataFrame(answer["lines"]).set_index("line_ref")
    key["picked"] = key.index.isin(LOCKED_PICKS)
    key["learned_score"] = sheet["learned_score"].reindex(key.index)
    nb.show(key[["outcome", "finding_type", "duty_at_stake_lcu", "learned_score", "picked"]],
            formats={"duty_at_stake_lcu": "{:,.0f}", "learned_score": "{:.2f}"}, highlight=key["outcome"] == "finding")
    hits = int(key.loc[key["picked"], "outcome"].eq("finding").sum())
    duty = key.loc[key["picked"] & key["outcome"].eq("finding"), "duty_at_stake_lcu"].sum()
    top5 = key.sort_values("learned_score", ascending=False).head(5)
    model_hits = int(top5["outcome"].eq("finding").sum())
    nb.finding(f"Your group found {hits} of 5 (duty at stake LCU {duty:,.0f}); a random pick finds "
               f"{answer['random_benchmark_hits_of_5']} of 5 (random, expected: 30% on this enriched set; real-world: 1 to 8%). "
               f"The learned score's top five, {', '.join(top5.index)}, hold {model_hits} findings.")
    calibration_line()
    for item in answer["explanations"]:
        print(f"- {item}")
    if nb.HINTS_USED:
        print("Hints used:", nb.HINTS_USED)

## 6 · For your analysts

The same pipeline with the code shown and the technical names. The four layers of the deck:

| Layer | In this notebook | Technical name |
| --- | --- | --- |
| Data | section 1, `load_tables`, the six-field check | extract, data dictionary, data-quality monitors |
| Signals | section 2, the columns of `features_lines` | features: robust ratio to a reference class with a fallback hierarchy, tenure, cheapest-code share drift, text-code disagreement (character n-grams), origin plausibility and exporter fan-out, amendment rate |
| Score | section 3, `learned_score`, `oddness_score`, `robust_z` | static-reference rule, robust z-score (median and MAD) and its own-history version, anomaly score (isolation forest), supervised model (gradient boosting) with feature contributions; precision at a 5% budget on a random control group; expiry (precision by month) |
| Decision | section 4 | the inspection list, the budget, the outcome record (label) written back |

**The extract to ask for.** The same tables, from the clearance system in the room: one line per
declaration item with the importer code and tax ID as keyed, the goods code, description, quantity,
unit, net weight, FOB, freight, insurance and customs value, the origin and consignment country, the
seller, the broker, the office, the lodgement date, the status history, the lane and the control
outcome. ASYCUDA World gives one extract format and one selectivity module (signals and rules become
ASYCUDA risk criteria); eCMS extracts need an IT request rather than a standard report; VNACCS/VCIS
holds the selectivity criteria in VCIS and the trader history separately from clearance; e-Customs
with a National Single Window serves extracts through the ICT bureau. Whichever it is: the outcome
of every control must be written back to the line, and a random share must be drawn before the rules
run.

**Your own extract.** Point `DATA_URL` (section 0) at a folder holding the same CSV files, or rename
your columns first with a mapping such as
`{"DECL_NO": "decl_id", "ITEM_NO": "line_no", "HS_CODE": "hs8", "CIF_VALUE": "customs_value_lcu", "NET_MASS": "net_kg", "IMPORTER_TIN": "tin_customs", "CTY_ORIGIN": "origin"}`.
The methods are in the `workshop_methods` package: `peer` (the peer group and its fallback), `signals`,
`ladder`, `newideas`; the precomputed columns are documented in `dictionary.md` beside the data.

**Keep a random share.** A model learns only from the lines someone chose to open. Draw the random
share before the rules run, record every outcome, and let the committee decide its size (the one study
that measured it found 10-20% exploration best where patterns drift).

**Further reading (titles only).** WCO Revenue Package; WCO Customs Risk Management Compendium;
IMF Technical Notes and Manuals 2024/006 (Aslett, González, Hadwick, Hamilton, Hardy, Pérez) and
2025/013 (Aslett, Cantens, Chastel, Crown, Hamilton); Chalendard et al., World Bank WPS 9254 (2020);
Mai et al. (2021) on exploration in customs selection.

In [ ]:
#@title 6 · The whole pipeline in one cell
# 1. Data: the lines that pay duty, with the precomputed signals
lines = f[f["regime"].isin(HOME_USE)]
# 2. Signals: the columns an analyst would compute (here read)
X = lines[["robust_z", "share_of_median", "text_code_disagreement", "weight_per_unit_dev", "code_drift_score", "exporter_importers_12m", "new_importer"]]
# 3. Score: a model trained on the months 1-18 controlled lines, evaluated on the months 19-36 random draw
train = lines["period_month"] <= 18
seen = train & lines["outcome"].notna()
test = (lines["period_month"] > 18) & (lines["random_draw"] == 1)
score, why = ladder.learning_score(X[seen], lines.loc[seen, "outcome"] == "finding", X[test])
print(f"precision at a 5% budget: {ladder.hit_rate_at_budget(score, lines.loc[test, 'outcome'] == 'finding', 0.05):.0%} "
      f"(the precomputed learned_score: {ladder.hit_rate_at_budget(lines.loc[test, 'learned_score'], lines.loc[test, 'outcome'] == 'finding', 0.05):.0%})")
# 4. Decision: the top 5% with their reasons, to be written back with the outcome
shortlist = lines[test].assign(score=score, reasons=why["reasons"].map(lambda r: ", ".join(x["signal"] for x in r))).nlargest(int(test.sum() * 0.05), "score")
print(shortlist[["line_id", "importer_id", "hs8", "score", "reasons"]].head(10).to_string(index=False))